In [15]:
import pandas as pd
import numpy as np
import requests

df = pd.read_json("../data/auto.json")
pd.options.display.float_format = '{:.2f}'.format

sample_size = 200
random_state = 21
unique_combinations = df[['CarNumber', 'Make', 'Model']].drop_duplicates()
sample_combinations = unique_combinations.sample(n=min(len(unique_combinations), sample_size), 
                                             replace=len(unique_combinations)<sample_size, random_state=random_state)
sample_refunds = df['Refund'].sample(n=len(sample_combinations), random_state=random_state, replace=True).reset_index(drop=True)
sample_fines = df['Fines'].sample(n=len(sample_combinations), random_state=random_state, replace=True).reset_index(drop=True)
sample_df = sample_combinations.reset_index(drop=True).copy()
sample_df['Refund'] = sample_refunds
sample_df['Fines'] = sample_fines

concatenated_df = pd.concat([df, sample_df], ignore_index=True, axis=0)

np.random.seed(21)
years = np.random.randint(1980, 2020, size=len(concatenated_df))
concatenated_df['Year'] = years
fines = concatenated_df
# print(fines)

surnames_data = pd.read_json("../../datasets/surname.json")
surnames_data.columns = surnames_data.iloc[0]
surnames_data = surnames_data[1:].reset_index(drop=True)
surnames_data['SURNAME'] = surnames_data['NAME'].str.replace(r'[^\w\s]', '', regex=True)
surnames = surnames_data['SURNAME'].dropna().unique()
# print(surnames)

unique_car_numbers = sample_df['CarNumber'].unique()
np.random.seed(21)
selected_surnames = np.random.choice(surnames, size=len(unique_car_numbers), replace=len(unique_car_numbers)>len(surnames))

owners = pd.DataFrame({
    'CarNumber': unique_car_numbers,
    'SURNAME': selected_surnames
})

new_fines = pd.DataFrame({
    'CarNumber': ['NEW001RUS', 'NEW002RUS', 'NEW003RUS', 'NEW004RUS', 'NEW005RUS'],
    'Make': ['Tesla', 'Tesla', 'Lada', 'Lada', 'Kia'],
    'Model': ['Model S', 'Model 3', 'Granta', 'Vesta', 'Rio'],
    'Refund': [2, 1, 2, 1, 2],
    'Fines': [5000.0, 7500.0, 3000.0, 4500.0, 6000.0],
    'Year': [2018, 2020, 2015, 2017, 2019]
})

fines = pd.concat([fines, new_fines], ignore_index=True)

owners = owners.iloc[:-20]
new_owners = pd.DataFrame({
    'CarNumber': ['OWNER001RUS', 'OWNER002RUS', 'OWNER003RUS'],
    'SURNAME': ['Smith', 'Johnson', 'Williams']
})
owners = pd.concat([owners, new_owners], ignore_index=True)

inner_join = pd.merge(fines, owners, on='CarNumber', how='inner')
outer_join = pd.merge(fines, owners, on='CarNumber', how='outer')
left_join = pd.merge(fines, owners, on='CarNumber', how='left')
right_join = pd.merge(fines, owners, on='CarNumber', how='right')

pivot_table =  fines.pivot_table(values='Fines', index=['Make', 'Model'], columns='Year', aggfunc='sum', fill_value=0)
print(pivot_table.iloc[:5, :5])

fines.to_csv('../data/fines.csv', index=False)
owners.to_csv('../data/owners.csv', index=False)

Year            1980      1981      1982     1983      1984
Make Model                                                 
Audi            0.00      0.00      0.00     0.00      0.00
BMW             0.00      0.00      0.00     0.00      0.00
Ford Focus  65794.59 396189.17 140383.76 62600.00 105994.59
     Mondeo     0.00      0.00      0.00     0.00      0.00
Kia  Rio        0.00      0.00      0.00     0.00      0.00
